# Experiment: GW Corner And Matrix Plot Distributions

Objective:
- Generate publication-style parameter distribution figures for the BNS and NSBH injection catalogs.
- Keep the 6D corner plot for each population.
- Add a 3x3 matrix figure for each population with six 1D histograms and three 2D density panels.


In [1]:
import os
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

from pathlib import Path

import corner
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path(f"{_BASE}/gw-kn-multimodal")
BNS_CSV = BASE / "dataset/O5_sim_bns_train/injections_final.csv"
NSBH_CSV = BASE / "dataset/O5_sim_nsbh_train/injections_pos.csv"
OUTDIR = BASE / "figures" / "gw_params_plots"
OUTDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 18,
    "axes.labelsize": 21,
    "axes.titlesize": 21,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "figure.dpi": 150,
})

BNS_CSV, NSBH_CSV, OUTDIR

(PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/dataset/O5_sim_bns_train/injections_final.csv'),
 PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/dataset/O5_sim_nsbh_train/injections_pos.csv'),
 PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots'))

## Plan

- Load the BNS and NSBH injection catalogs.
- Build standardized parameter frames for the two populations.
- Use `corner.corner()` for the full 6D distribution view.
- Use a custom `3x3` matrix plot for the compact summary view.


In [2]:
bns = pd.read_csv(BNS_CSV)
nsbh = pd.read_csv(NSBH_CSV)

pd.DataFrame([
    {"dataset": "BNS", "rows": len(bns), "columns": len(bns.columns)},
    {"dataset": "NSBH", "rows": len(nsbh), "columns": len(nsbh.columns)},
])

,dataset,rows,columns
0,BNS,27553,29
1,NSBH,15978,27


In [3]:
def plot_corner(frame, label_map, title, outpath, color):
    labels = [label_map[col] for col in frame.columns]
    fig = corner.corner(
        frame.to_numpy(),
        labels=labels,
        bins=35,
        color=color,
        smooth=1.0,
        fill_contours=True,
        plot_datapoints=False,
        show_titles=False,
        title_fmt=".2f",
        quantiles=[0.16, 0.5, 0.84],
        label_kwargs={"fontsize": 21},
        title_kwargs={"fontsize": 17},
        hist_kwargs={"density": True, "alpha": 0.9},
    )
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_matrix_grid(frame, label_map, title, outpath, color, density_pairs, cmap):
    fig, axes = plt.subplots(3, 3, figsize=(15, 13))
    hist_columns = list(frame.columns)

    for ax, key in zip(axes[:2].flat, hist_columns):
        values = frame[key].to_numpy()
        ax.hist(values, bins=35, color=color, alpha=0.85, edgecolor="white", linewidth=0.7)
        ax.set_xlabel(label_map[key])
        ax.set_ylabel("Count")
        ax.grid(alpha=0.2, linestyle=":")

    for ax, (x_key, y_key, panel_title) in zip(axes[2], density_pairs):
        ax.hexbin(
            frame[x_key].to_numpy(),
            frame[y_key].to_numpy(),
            gridsize=45,
            mincnt=1,
            bins="log",
            cmap=cmap,
            linewidths=0,
        )
        ax.set_xlabel(label_map[x_key])
        ax.set_ylabel(label_map[y_key])

    fig.tight_layout(rect=[0, 0, 1, 0.975])
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def build_bns_frame(df):
    frame = df[["mass1", "mass2", "spin1z", "spin2z", "distance", "inclination"]].dropna().rename(
        columns={
            "mass1": "m1",
            "mass2": "m2",
            "spin1z": "chi1",
            "spin2z": "chi2",
            "distance": "dL",
            "inclination": "inclination",
        }
    )
    label_map = {
        "m1": r"$m_1\,[M_\odot]$",
        "m2": r"$m_2\,[M_\odot]$",
        "chi1": r"$\chi_1$",
        "chi2": r"$\chi_2$",
        "dL": r"$d_L\,[\mathrm{Mpc}]$",
        "inclination": r"$\iota\,[\mathrm{rad}]$",
    }
    density_pairs = [
        ("m1", "m2", r"$m_1$ vs $m_2$"),
        ("m1", "chi1", r"$m_1$ vs $\chi_1$"),
        ("dL", "inclination", r"$d_L$ vs $\iota$"),
    ]
    return frame, label_map, density_pairs


def build_nsbh_frame(df):
    base = df[["mass1", "mass2", "spin1z", "spin2z", "distance", "inclination"]].dropna()
    mass1 = base["mass1"].to_numpy()
    mass2 = base["mass2"].to_numpy()
    spin1 = base["spin1z"].to_numpy()
    spin2 = base["spin2z"].to_numpy()

    frame = pd.DataFrame(
        {
            "m_ns": np.minimum(mass1, mass2),
            "m_bh": np.maximum(mass1, mass2),
            "chi_ns": np.where(mass1 < mass2, spin1, spin2),
            "chi_bh": np.where(mass1 >= mass2, spin1, spin2),
            "dL": base["distance"].to_numpy(),
            "inclination": base["inclination"].to_numpy(),
        }
    )
    label_map = {
        "m_ns": r"$m_{\mathrm{NS}}\,[M_\odot]$",
        "m_bh": r"$m_{\mathrm{BH}}\,[M_\odot]$",
        "chi_ns": r"$\chi_{\mathrm{NS}}$",
        "chi_bh": r"$\chi_{\mathrm{BH}}$",
        "dL": r"$d_L\,[\mathrm{Mpc}]$",
        "inclination": r"$\iota\,[\mathrm{rad}]$",
    }
    density_pairs = [
        ("m_ns", "m_bh", r"$m_{\mathrm{NS}}$ vs $m_{\mathrm{BH}}$"),
        ("m_bh", "chi_bh", r"$m_{\mathrm{BH}}$ vs $\chi_{\mathrm{BH}}$"),
        ("dL", "inclination", r"$d_L$ vs $\iota$"),
    ]
    return frame, label_map, density_pairs

## Build Standardized Parameter Frames

For NSBH, the lighter compact object is mapped to the NS axis and the heavier one to the BH axis on an event-by-event basis.


In [4]:
bns_frame, bns_labels, bns_pairs = build_bns_frame(bns)
nsbh_frame, nsbh_labels, nsbh_pairs = build_nsbh_frame(nsbh)

{
    "bns_shape": bns_frame.shape,
    "nsbh_shape": nsbh_frame.shape,
    "bns_columns": list(bns_frame.columns),
    "nsbh_columns": list(nsbh_frame.columns),
}

{'bns_shape': (27553, 6),
 'nsbh_shape': (15978, 6),
 'bns_columns': ['m1', 'm2', 'chi1', 'chi2', 'dL', 'inclination'],
 'nsbh_columns': ['m_ns', 'm_bh', 'chi_ns', 'chi_bh', 'dL', 'inclination']}

## Generate Corner Plots


In [5]:
bns_corner = OUTDIR / "bns_params_corner.png"
nsbh_corner = OUTDIR / "nsbh_params_corner.png"

plot_corner(bns_frame, bns_labels, "BNS Injection Parameters", bns_corner, "#1f77b4")
plot_corner(nsbh_frame, nsbh_labels, "NSBH Injection Parameters", nsbh_corner, "#d62728")

{
    "bns_corner": str(bns_corner),
    "nsbh_corner": str(nsbh_corner),
}

{'bns_corner': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/bns_params_corner.png',
 'nsbh_corner': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/nsbh_params_corner.png'}

## Generate 3x3 Matrix Plots


In [6]:
bns_matrix = OUTDIR / "bns_params_matrix.png"
nsbh_matrix = OUTDIR / "nsbh_params_matrix.png"

plot_matrix_grid(bns_frame, bns_labels, "Parameter distributions of BNS", bns_matrix, "#1f77b4", bns_pairs, "Blues")
plot_matrix_grid(nsbh_frame, nsbh_labels, "Parameter distributions of NSBH", nsbh_matrix, "#d62728", nsbh_pairs, "Reds")

{
    "bns_matrix": str(bns_matrix),
    "nsbh_matrix": str(nsbh_matrix),
}

{'bns_matrix': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/bns_params_matrix.png',
 'nsbh_matrix': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/nsbh_params_matrix.png'}

In [7]:
result = {
    "bns_rows_used": int(bns_frame.shape[0]),
    "nsbh_rows_used": int(nsbh_frame.shape[0]),
    "bns_corner": str(bns_corner),
    "nsbh_corner": str(nsbh_corner),
    "bns_matrix": str(bns_matrix),
    "nsbh_matrix": str(nsbh_matrix),
}
result

{'bns_rows_used': 27553,
 'nsbh_rows_used': 15978,
 'bns_corner': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/bns_params_corner.png',
 'nsbh_corner': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/nsbh_params_corner.png',
 'bns_matrix': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/bns_params_matrix.png',
 'nsbh_matrix': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/nsbh_params_matrix.png'}

## Results

- The notebook now includes both the corner-plot method and the 3x3 matrix-plot method.
- Each population produces two figures: one corner plot and one matrix plot.
- The matrix plot uses the same parameter mapping and figure style as the current Python script.


## Mass Distributions from Test Population Samplers

Recreate the original test-population mass samples from the `observing-scenarios-simulations` generator scripts. For BNS, plot the raw recycled and slow component distributions before they are sorted into `mass1=max(...)` and `mass2=min(...)`. For NSBH, use a joint mass-density plot with marginal histograms so the narrow neutron-star range and the broad power-law black-hole tail are both visible.


In [8]:
import importlib.util
import sys

SIM_DIR = Path(f"{_BASE}/observing-scenarios-simulations")
SIM_SCRIPT_DIR = SIM_DIR / "scripts"
if str(SIM_SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SIM_SCRIPT_DIR))


def _load_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


bns_test_dist = _load_module(
    "bns_test_distribution",
    SIM_SCRIPT_DIR / "generate_bns_test_distribution.py",
)
nsbh_test_dist = _load_module(
    "nsbh_test_distribution",
    SIM_SCRIPT_DIR / "generate_nsbh_test_distribution.py",
)

TEST_NSAMPLES = 100_000
TEST_SEED = 42

# BNS original component samples. These are the paired recycled/slow masses
# before generate_bns_test_distribution sorts them into mass1/mass2.
BNS_RECYCLED_MASS_LO = bns_test_dist.DEFAULT_RECYCLED_MASS_MIN
BNS_RECYCLED_MASS_HI = bns_test_dist.DEFAULT_RECYCLED_MASS_MAX
BNS_SLOW_MASS_LO = bns_test_dist.DEFAULT_SLOW_MASS_MIN
BNS_SLOW_MASS_HI = bns_test_dist.DEFAULT_SLOW_MASS_MAX
BNS_MASS_LO = min(BNS_RECYCLED_MASS_LO, BNS_SLOW_MASS_LO)
BNS_MASS_HI = max(BNS_RECYCLED_MASS_HI, BNS_SLOW_MASS_HI)

rng_bns = np.random.default_rng(TEST_SEED)
bns_recycled_mass = bns_test_dist._sample_recycled_mass(
    rng_bns,
    TEST_NSAMPLES,
    BNS_RECYCLED_MASS_LO,
    BNS_RECYCLED_MASS_HI,
)
bns_slow_mass = bns_test_dist._sample_slow_mass(
    rng_bns,
    TEST_NSAMPLES,
    BNS_SLOW_MASS_LO,
    BNS_SLOW_MASS_HI,
)

# NSBH original component samples from the custom test generator:
# mass1 is the BH draw and mass2 is the NS draw.
NSBH_NS_MASS_LO = nsbh_test_dist.DEFAULT_NS_MASS_MIN
NSBH_NS_MASS_HI = nsbh_test_dist.DEFAULT_NS_MASS_MAX
NSBH_BH_MASS_LO = nsbh_test_dist.DEFAULT_BH_MASS_MIN
NSBH_BH_MASS_HI = nsbh_test_dist.DEFAULT_BH_MASS_MAX

nsbh_table = nsbh_test_dist.generate_distribution(
    nsamples=TEST_NSAMPLES,
    seed=TEST_SEED,
    mass_method="custom",
)
nsbh_m1 = np.asarray(nsbh_table["mass1"], dtype=np.float64)
nsbh_m2 = np.asarray(nsbh_table["mass2"], dtype=np.float64)
nsbh_bh_mass = nsbh_m1
nsbh_ns_mass = nsbh_m2

{
    "bns_samples": int(len(bns_recycled_mass)),
    "bns_recycled_range": (float(bns_recycled_mass.min()), float(bns_recycled_mass.max())),
    "bns_slow_range": (float(bns_slow_mass.min()), float(bns_slow_mass.max())),
    "nsbh_samples": int(len(nsbh_m1)),
    "nsbh_m1_bh_range": (float(nsbh_bh_mass.min()), float(nsbh_bh_mass.max())),
    "nsbh_m2_ns_range": (float(nsbh_ns_mass.min()), float(nsbh_ns_mass.max())),
}


{'bns_samples': 100000,
 'bns_recycled_range': (1.0005925943781908, 2.012746957312365),
 'bns_slow_range': (1.0001190664058532, 2.0499973469928867),
 'nsbh_samples': 100000,
 'nsbh_m1_bh_range': (2.050015449523926, 19.996530532836914),
 'nsbh_m2_ns_range': (1.0000238418579102, 2.0499978065490723)}

In [9]:
from matplotlib.ticker import NullFormatter

MASS_TITLE_FONTSIZE = 22
MASS_LABEL_FONTSIZE = 22
MASS_TICK_FONTSIZE = 18
MASS_LEGEND_FONTSIZE = 19
# BNS: original component histograms before mass sorting into m1/m2.
fig, ax = plt.subplots(figsize=(8, 5.5))
bins_bns = np.linspace(BNS_MASS_LO, BNS_MASS_HI, 45)
ax.hist(
    bns_recycled_mass,
    bins=bins_bns,
    density=True,
    histtype="step",
    color="#2563EB",
    linewidth=2.2,
    label="Recycled NS",
)
ax.hist(
    bns_slow_mass,
    bins=bins_bns,
    density=True,
    histtype="step",
    color="#DC2626",
    linewidth=2.2,
    label="Slow NS",
)
ax.set_xlabel(r"Mass $(M_\odot)$", fontsize=MASS_LABEL_FONTSIZE)
ax.set_ylabel("Probability density", fontsize=MASS_LABEL_FONTSIZE)
ax.legend(frameon=False, loc="upper right", fontsize=MASS_LEGEND_FONTSIZE)
ax.set_xlim(BNS_MASS_LO, BNS_MASS_HI)
ax.grid(alpha=0.2, linestyle=":")
ax.tick_params(axis="both", labelsize=MASS_TICK_FONTSIZE)
fig.tight_layout()
bns_mass_path = OUTDIR / "bns_test_mass_distribution.png"
fig.savefig(bns_mass_path, dpi=300, bbox_inches="tight")
fig.savefig(OUTDIR / "bns_test_mass_distribution.pdf", bbox_inches="tight")
plt.close(fig)

# NSBH: joint component mass density with marginal histograms. The BH axis uses
# a log scale to resolve the low-mass concentration and the long power-law tail.
fig = plt.figure(figsize=(9.2, 7.6))
gs = fig.add_gridspec(
    2,
    3,
    width_ratios=(4.4, 1.25, 0.18),
    height_ratios=(1.25, 4.2),
    wspace=0.16,
    hspace=0.14,
)
ax_joint = fig.add_subplot(gs[1, 0])
ax_top = fig.add_subplot(gs[0, 0], sharex=ax_joint)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)
cax = fig.add_subplot(gs[1, 2])

ns_bins = np.linspace(NSBH_NS_MASS_LO, NSBH_NS_MASS_HI, 45)
bh_bins = np.geomspace(NSBH_BH_MASS_LO, NSBH_BH_MASS_HI, 45)
hexbin_probability = np.full(nsbh_ns_mass.shape, 1.0 / nsbh_ns_mass.size, dtype=np.float64)

hb = ax_joint.hexbin(
    nsbh_ns_mass,
    nsbh_bh_mass,
    C=hexbin_probability,
    reduce_C_function=np.sum,
    gridsize=(28, 32),
    mincnt=1,
    bins="log",
    xscale="linear",
    yscale="log",
    cmap="magma_r",
    linewidths=0,
    rasterized=True,
)
ax_joint.set_xlabel(r"$m_{\mathrm{NS}}\,(M_\odot)$", fontsize=MASS_LABEL_FONTSIZE)
ax_joint.set_ylabel(r"$m_{\mathrm{BH}}\,(M_\odot)$", fontsize=MASS_LABEL_FONTSIZE)
ax_joint.set_xlim(NSBH_NS_MASS_LO, NSBH_NS_MASS_HI)
ax_joint.set_ylim(NSBH_BH_MASS_LO, NSBH_BH_MASS_HI)
ax_joint.set_yscale("log")
bh_ticks = [2.05, 3, 5, 10, 20]
ax_joint.set_yticks(bh_ticks)
ax_joint.set_yticklabels(["2.05", "3", "5", "10", "20"])
ax_joint.yaxis.set_minor_formatter(NullFormatter())
ax_joint.grid(alpha=0.2, linestyle=":")
ax_joint.tick_params(axis="both", labelsize=MASS_TICK_FONTSIZE)

ax_top.hist(
    nsbh_ns_mass,
    histtype="step",
    bins=ns_bins,
    density=True,
    edgecolor="#2563EB",
    alpha=0.85,
    linewidth=1,
)
ax_top.set_ylabel("Density", fontsize=MASS_LABEL_FONTSIZE)
ax_top.grid(alpha=0.2, linestyle=":")
ax_top.tick_params(axis="x", labelbottom=False, labelsize=MASS_TICK_FONTSIZE)
ax_top.tick_params(axis="y", labelsize=MASS_TICK_FONTSIZE)

ax_right.hist(
    nsbh_bh_mass,
    histtype="step",
    bins=bh_bins,
    density=True,
    orientation="horizontal",
    edgecolor="#DC2626",
    alpha=0.85,
    linewidth=1,
)
ax_right.set_xscale("log")
ax_right.set_xlabel("Density", fontsize=MASS_LABEL_FONTSIZE)
ax_right.grid(alpha=0.2, linestyle=":")
ax_right.tick_params(axis="y", labelleft=False, labelsize=MASS_TICK_FONTSIZE)
ax_right.tick_params(axis="x", labelsize=MASS_TICK_FONTSIZE)
ax_right.yaxis.set_minor_formatter(NullFormatter())

cbar = fig.colorbar(hb, cax=cax)
cbar.set_label("Probability per hexbin", fontsize=MASS_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=MASS_TICK_FONTSIZE)

nsbh_joint_path = OUTDIR / "nsbh_test_mass_joint_distribution.png"
fig.savefig(nsbh_joint_path, dpi=300, bbox_inches="tight")
fig.savefig(OUTDIR / "nsbh_test_mass_joint_distribution.pdf", bbox_inches="tight")
plt.close(fig)

{
    "bns_mass_distribution": str(bns_mass_path),
    "nsbh_mass_joint_distribution": str(nsbh_joint_path),
}


{'bns_mass_distribution': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/bns_test_mass_distribution.png',
 'nsbh_mass_joint_distribution': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/gw_params_plots/nsbh_test_mass_joint_distribution.png'}